# World Cup 2026 — Match Predictor
### Step 1: Data Loading & Preparation

In [ ]:
import pandas as pd
import numpy as np
from datetime import date

# pandas  → data manipulation (loading CSVs, filtering, merging)
# numpy   → math/array ops we'll need for weighting and model features
# date    → needed to compute how old each match is for recency weighting

## 1. Load the raw data

In [ ]:
results    = pd.read_csv('results.csv',      parse_dates=['date'])
shootouts  = pd.read_csv('shootouts.csv',    parse_dates=['date'])
fnames     = pd.read_csv('former_names.csv', parse_dates=['start_date', 'end_date'])

print(f'Results:   {len(results):,} matches')
print(f'Shootouts: {len(shootouts):,} matches')
print(f'Name mappings: {len(fnames)}')
results.head()

## 2. Normalize team names using former_names

Some countries renamed over time (e.g. Zaire → DR Congo).
We map every historical name to its current name so team stats
are correctly aggregated across all years.

In [ ]:
name_map = dict(zip(fnames['former'], fnames['current']))

def normalize_team(name):
    return name_map.get(name, name)

for col in ['home_team', 'away_team']:
    results[col]   = results[col].map(normalize_team)
    shootouts[col] = shootouts[col].map(normalize_team)
shootouts['winner'] = shootouts['winner'].map(normalize_team)

print('Team names normalized.')

## 3. Merge shootout results into results

results.csv records the 90-min score only. When a knockout game
ends in a draw and goes to penalties, we need to know who actually
won — that's what shootouts.csv adds.

`shootout_winner` will be NaN for all regular (non-penalty) matches.

In [ ]:
shootouts_slim = shootouts[['date', 'home_team', 'away_team', 'winner']].rename(
    columns={'winner': 'shootout_winner'}
)

df = results.merge(shootouts_slim, on=['date', 'home_team', 'away_team'], how='left')

def true_winner(row):
    if pd.notna(row['shootout_winner']):
        return row['shootout_winner']
    elif row['home_score'] > row['away_score']:
        return row['home_team']
    elif row['away_score'] > row['home_score']:
        return row['away_team']
    else:
        return 'Draw'

df['winner'] = df.apply(true_winner, axis=1)

print(f'Matches with shootout data: {df["shootout_winner"].notna().sum()}')

## 4. Extract WC 2026 teams directly from the data

Rather than hardcoding the 48 teams by hand (and risking errors),
we pull them straight from the dataset — any team that appears
in a 'FIFA World Cup' match in 2026.

In [ ]:
wc2026_matches = df[
    (df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2026)
]

WC2026_TEAMS = set(
    wc2026_matches['home_team'].tolist() + wc2026_matches['away_team'].tolist()
)

print(f'WC 2026 teams found: {len(WC2026_TEAMS)}')
print(sorted(WC2026_TEAMS))

## 5. Split into training data and test data

**Train:** all historical matches between WC 2026 teams, EXCLUDING the WC 2026 games themselves.  
**Test:** only WC 2026 games — we'll use these to evaluate how accurate the model is.

This is the standard train/test split for backtesting: train on the past, predict the present.

In [ ]:
# Both teams must be WC 2026 participants
both_wc_teams = df['home_team'].isin(WC2026_TEAMS) & df['away_team'].isin(WC2026_TEAMS)

# WC 2026 games = the test set
is_wc2026 = (df['tournament'] == 'FIFA World Cup') & (df['date'].dt.year == 2026)

train = df[both_wc_teams & ~is_wc2026].copy().sort_values('date').reset_index(drop=True)
test  = df[is_wc2026].copy().sort_values('date').reset_index(drop=True)

print(f'Training set: {len(train):,} matches (all history, WC 2026 teams only)')
print(f'Test set:     {len(test):,} WC 2026 matches')
print(f'Date range of training data: {train["date"].min().date()} → {train["date"].max().date()}')

## 6. Recency weighting (applied to training data)

All historical data is kept, but older matches carry less weight:
- Last 3 years  (2023–2026): weight = 3.0
- Last 3–8 years (2018–2022): weight = 2.0
- Older (pre-2018):           weight = 1.0

In [ ]:
TODAY = pd.Timestamp(date.today())

def recency_weight(match_date):
    days_ago = (TODAY - match_date).days
    if days_ago <= 365 * 3:
        return 3.0
    elif days_ago <= 365 * 8:
        return 2.0
    else:
        return 1.0

train['weight'] = train['date'].apply(recency_weight)

print('Weight distribution in training set:')
print(train['weight'].value_counts().sort_index())

## 7. Sanity check — head-to-head lookup

In [ ]:
def head_to_head(team_a, team_b):
    """Return all training matches between two teams (regardless of home/away)."""
    mask = (
        ((train['home_team'] == team_a) & (train['away_team'] == team_b)) |
        ((train['home_team'] == team_b) & (train['away_team'] == team_a))
    )
    return train[mask][['date', 'home_team', 'away_team', 'home_score',
                         'away_score', 'tournament', 'winner', 'weight']]

h2h = head_to_head('Brazil', 'Mexico')
print(f'Brazil vs Mexico — all time (training): {len(h2h)} matches')
h2h

## Data is ready

- `train` — historical matches between WC 2026 teams, with recency weights. Use this to build the model.
- `test` — WC 2026 games. Use these to measure accuracy.

Next steps:
1. Feature engineering — compute weighted team stats (attack strength, defense, form)
2. Model training — fit a classifier on `train`
3. Backtesting — predict `test` outcomes and compare to real results
4. Future prediction — input upcoming fixtures